In [3]:
# -*- coding: utf-8 -*-
"""
backend10_product_change_timing.ipynb (single-run, Jupyter)

[확정 사항]
- backend: 10
- prod_day=20260128, shift_type=day (우선 고정)
- 대상: station in ('Vision1','Vision2'), remark in ('PD','Non-PD')
- end_time: 기본 HH:MI:SS (ms 없음). (안전장치로 ms가 와도 0.5 반올림 지원)
- 이벤트 정의(핵심):
  * end_time(=end_ts) DESC(최신->과거) 정렬에서
  * "위(더 최신)" remark != "아래(더 과거)" remark 이면 이벤트 1개 생성
    - at_time     = 위 행 end_time
    - from_remark = 아래 행 remark
    - to_remark   = 위 행 remark
  * station별로 독립 처리, 하루 몇 회든지 모두 기록
  * 윈도우 시작 시점(08:30:00) 관련 이벤트는 만들지 않음(전환만 기록)
- updated_at: 이벤트별 계산 시각(행 생성 시각, KST)
"""

from __future__ import annotations

from datetime import datetime, date, timedelta
from decimal import Decimal, ROUND_HALF_UP
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

# =========================
# 1) DB 설정
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
TABLE  = "fct_vision_testlog_txt_processing_history"

KST = ZoneInfo("Asia/Seoul")

# =========================
# 2) 실행 파라미터(우선 고정)
# =========================
PROD_DAY   = "20260128"
SHIFT_TYPE = "day"  # 'day' or 'night'

# =========================
# 3) 유틸
# =========================
def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(url, pool_size=1, max_overflow=0, pool_pre_ping=True, pool_recycle=1800)

def parse_prod_day(prod_day: str) -> date:
    return date(int(prod_day[0:4]), int(prod_day[4:6]), int(prod_day[6:8]))

def shift_window(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    d = parse_prod_day(prod_day)
    if shift_type == "day":
        start = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
        end   = datetime(d.year, d.month, d.day, 20, 29, 59, tzinfo=KST)
    elif shift_type == "night":
        start = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
        d2 = d + timedelta(days=1)
        end   = datetime(d2.year, d2.month, d2.day, 8, 29, 59, tzinfo=KST)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end

def normalize_time_hhmmss(t: str) -> str:
    """
    입력: "HH:MI:SS" 또는 "HH:MI:SS.xxx"
    출력: "HH:MI:SS"
    - ms가 있으면 0.5 기준 반올림(half-up)
      예: 12:01:08.50 -> 12:01:09
    """
    t = (t or "").strip()
    if not t:
        return t
    if "." not in t:
        return t

    hhmmss, frac = t.split(".", 1)
    frac = "".join(ch for ch in frac if ch.isdigit())
    if not frac:
        return hhmmss

    denom = Decimal(10) ** Decimal(len(frac))
    sec_fraction = (Decimal(frac) / denom)  # 0~<1

    h, m, s = [int(x) for x in hhmmss.split(":")]
    base = datetime(2000, 1, 1, h, m, s)

    # half-up: 0.5 이상이면 1초 올림
    add_one = int(sec_fraction.quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    base2 = base + timedelta(seconds=add_one)
    return base2.strftime("%H:%M:%S")

# =========================
# 4) 데이터 로드
# =========================
window_start, window_end = shift_window(PROD_DAY, SHIFT_TYPE)

# DB timestamp without tz 가정 -> tz 제거 후 문자열 파라미터로 전달
ws = window_start.replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S")
we = window_end.replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S")

sql = text(f"""
WITH base AS (
    SELECT
        end_day,
        end_time,
        station,
        remark,
        split_part(end_time, '.', 1) AS end_time_sec,
        to_timestamp(end_day || ' ' || split_part(end_time, '.', 1), 'YYYYMMDD HH24:MI:SS') AS end_ts
    FROM {SCHEMA}.{TABLE}
    WHERE station IN ('Vision1', 'Vision2')
      AND remark IN ('PD', 'Non-PD')
)
SELECT
    end_day,
    end_time_sec AS end_time,
    station,
    remark,
    end_ts
FROM base
WHERE end_ts >= to_timestamp(:ws, 'YYYY-MM-DD HH24:MI:SS')
  AND end_ts <= to_timestamp(:we, 'YYYY-MM-DD HH24:MI:SS')
ORDER BY station, end_ts DESC;
""")

engine = make_engine()
df = pd.read_sql(sql, engine, params={"ws": ws, "we": we})

# end_time 안전 정규화(원칙상 ms 없음이지만 안전장치)
if not df.empty:
    df["end_time"] = df["end_time"].astype(str).map(normalize_time_hhmmss)

print(f"[INFO] window={ws} ~ {we} | rows={len(df)}")
df.head(10)

[INFO] window=2026-01-28 08:30:00 ~ 2026-01-28 20:29:59 | rows=3221


,end_day,end_time,station,remark,end_ts
0,20260128,20:07:47,Vision1,PD,2026-01-28 11:07:47+00:00
1,20260128,20:07:35,Vision1,PD,2026-01-28 11:07:35+00:00
2,20260128,20:06:42,Vision1,PD,2026-01-28 11:06:42+00:00
3,20260128,20:06:22,Vision1,PD,2026-01-28 11:06:22+00:00
4,20260128,20:06:04,Vision1,PD,2026-01-28 11:06:04+00:00
5,20260128,20:05:43,Vision1,PD,2026-01-28 11:05:43+00:00
6,20260128,20:05:26,Vision1,PD,2026-01-28 11:05:26+00:00
7,20260128,20:05:06,Vision1,PD,2026-01-28 11:05:06+00:00
8,20260128,20:04:50,Vision1,PD,2026-01-28 11:04:50+00:00
9,20260128,20:04:29,Vision1,PD,2026-01-28 11:04:29+00:00


In [4]:
# =========================
# 5) 전환 이벤트 산출 (station별)
# =========================
events = []

if not df.empty:
    for station, g in df.groupby("station", sort=False):
        # end_ts DESC 보정
        g = g.sort_values(["end_ts"], ascending=[False], kind="mergesort").reset_index(drop=True)

        # 최신 -> 과거로 내려가며 i(위) vs i+1(아래)
        for i in range(len(g) - 1):
            cur_remark = g.at[i, "remark"]      # 위(더 최신)
            nxt_remark = g.at[i + 1, "remark"]  # 아래(더 과거)

            if cur_remark != nxt_remark:
                events.append({
                    "prod_day": PROD_DAY,
                    "shift_type": SHIFT_TYPE,
                    "station": station,
                    "at_time": g.at[i, "end_time"],   # 위쪽 행 end_time
                    "from_remark": nxt_remark,        # 아래쪽 remark
                    "to_remark": cur_remark,          # 위쪽 remark
                    "updated_at": datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S"),
                })

result = pd.DataFrame(
    events,
    columns=["prod_day","shift_type","station","at_time","from_remark","to_remark","updated_at"]
)

print(f"[INFO] events={len(result)}")
result

[INFO] events=2


,prod_day,shift_type,station,at_time,from_remark,to_remark,updated_at
0,20260128,day,Vision1,11:04:19,Non-PD,PD,2026-01-30 19:23:18
1,20260128,day,Vision2,11:04:54,Non-PD,PD,2026-01-30 19:23:18
